# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** et utiliser médiane, IQR, P90.
2. Écrire des requêtes Spark SQL avec CTE, DENSE_RANK, ROW_NUMBER, Pareto et LAG.
3. Construire 3 tables Gold pour Streamlit.
4. Compacter Silver avec OPTIMIZE.


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("J3_Analytics_SQL_Gold").getOrCreate()
SILVER_MAIN_PATH = "/lakehouse/silver/orders_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").first()[0])


## 1. Analyse d'Asymétrie : Moyenne vs Médiane & IQR
### Exercice 1.1 — Calculer COUNT, AVG, STDDEV, SKEWNESS, P25, P50, P75, P90, P95, MAX sur la valeur des commandes.


In [ ]:
df_dist_summary = spark.sql("""
SELECT COUNT(*) AS n, AVG(order_value) AS moyenne, STDDEV(order_value) AS ecart_type,
       SKEWNESS(order_value) AS asymetrie,
       PERCENTILE_APPROX(order_value, 0.25) AS p25,
       PERCENTILE_APPROX(order_value, 0.50) AS mediane,
       PERCENTILE_APPROX(order_value, 0.75) AS p75,
       PERCENTILE_APPROX(order_value, 0.90) AS p90,
       PERCENTILE_APPROX(order_value, 0.95) AS p95,
       MAX(order_value) AS maximum,
       PERCENTILE_APPROX(order_value, 0.75)-PERCENTILE_APPROX(order_value, 0.25) AS iqr
FROM v_silver WHERE order_value IS NOT NULL
""")
df_dist_summary.show(truncate=False)


**Justification statistique :** Une asymétrie positive et une moyenne supérieure à la médiane indiquent une queue vers les commandes chères. La médiane résiste aux valeurs extrêmes ; l'IQR (P75−P25) décrit la dispersion centrale. Le dashboard présente la médiane et P90, avec la moyenne en complément. La conclusion exacte dépend des résultats affichés.

## 2. Table Gold #1 : Classement Métier & Pareto
### Exercice 2.1 — Agrégation par catégorie, DENSE_RANK, part cumulée du chiffre d'affaires.


In [ ]:
GOLD_PARETO_PATH = "/lakehouse/gold/category_pareto"
df_gold_1 = spark.sql("""
WITH categories AS (
 SELECT COALESCE(NULLIF(TRIM(product_category), ''), 'inconnue') AS category,
        COUNT(*) AS nb_orders, SUM(order_value) AS metric_total,
        PERCENTILE_APPROX(order_value, 0.5) AS median_order_value,
        PERCENTILE_APPROX(order_value, 0.75)-PERCENTILE_APPROX(order_value, 0.25) AS iqr_order_value,
        AVG(CASE WHEN is_late = true THEN 1.0 ELSE 0.0 END) AS late_rate
 FROM v_silver WHERE order_value IS NOT NULL
 GROUP BY COALESCE(NULLIF(TRIM(product_category), ''), 'inconnue')
)
SELECT *, DENSE_RANK() OVER (ORDER BY metric_total DESC) AS revenue_rank,
       SUM(metric_total) OVER (ORDER BY metric_total DESC, category ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
         / NULLIF(SUM(metric_total) OVER (), 0) AS pareto_cumulative_share
FROM categories
""")
df_gold_1.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PARETO_PATH)
df_gold_1.show(15, truncate=False)


## 3. Table Gold #2 : Dynamique Temporelle & LAG
### Exercice 3.1 — Agrégation mensuelle, croissance mois sur mois.


In [ ]:
GOLD_TIME_PATH = "/lakehouse/gold/monthly_trends"
df_gold_2 = spark.sql("""
WITH monthly AS (
 SELECT DATE_FORMAT(purchase_date, 'yyyy-MM') AS period,
        COUNT(*) AS nb_orders, SUM(order_value) AS revenue,
        PERCENTILE_APPROX(order_value, 0.5) AS median_order_value,
        AVG(CASE WHEN is_late = true THEN 1.0 ELSE 0.0 END) AS late_rate
 FROM v_silver WHERE purchase_date IS NOT NULL AND order_value IS NOT NULL
 GROUP BY DATE_FORMAT(purchase_date, 'yyyy-MM')
), lagged AS (
 SELECT *, LAG(revenue, 1) OVER (ORDER BY period) AS previous_revenue
 FROM monthly
)
SELECT *, CASE WHEN previous_revenue > 0 THEN (revenue-previous_revenue)/previous_revenue END AS revenue_growth
FROM lagged ORDER BY period
""")
df_gold_2.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_TIME_PATH)
df_gold_2.show(20, truncate=False)


## 4. Table Gold #3 : Top catégorie par État & OPTIMIZE
### Exercice 4.1 — ROW_NUMBER() PARTITION BY état et compaction Silver.


In [ ]:
GOLD_REGIONAL_PATH = "/lakehouse/gold/top_category_by_state"
df_gold_3 = spark.sql("""
WITH state_categories AS (
 SELECT COALESCE(customer_state, 'inconnu') AS customer_state,
        COALESCE(product_category, 'inconnue') AS category,
        COUNT(*) AS nb_orders, SUM(order_value) AS revenue,
        AVG(CASE WHEN is_late = true THEN 1.0 ELSE 0.0 END) AS late_rate
 FROM v_silver WHERE order_value IS NOT NULL
 GROUP BY COALESCE(customer_state, 'inconnu'), COALESCE(product_category, 'inconnue')
), ranked AS (
 SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_state ORDER BY revenue DESC, category) AS row_num
 FROM state_categories
)
SELECT customer_state, category, nb_orders, revenue, late_rate FROM ranked WHERE row_num=1
""")
df_gold_3.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_REGIONAL_PATH)
df_gold_3.show(30, truncate=False)
print("OPTIMIZE Silver :")
spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)
print("La compaction réduit les micro-fichiers et les métadonnées gérées par HDFS NameNode.")
